# Reproduce Study 1

The main analysis dataset is [data/regression_data.csv](../data/regression_data.csv), and the main Python analysis script is [run.py](run.py). This notebook runs the same analysis with short explanations. Install `../requirements.txt`, then choose **Run all**.

This notebook loads the main data, prepares the comparison datasets, runs the analyses, and displays the paper's tables and empirical figure. No API key or original computer path is needed.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import sys
import json
import pandas as pd
from IPython.display import display, Markdown, Image

# Work when the notebook is opened in scripts/ or in the main folder.
repository = Path.cwd()
if not (repository / "data").is_dir():
    repository = repository.parent
if not (repository / "data").is_dir():
    raise FileNotFoundError("Please open the complete Study 1 folder, including data/ and scripts/.")
sys.path.insert(0, str(repository))

from scripts.supporting.prepare_data import build_data, prepare_analysis_data
from scripts.supporting.analysis import run_analysis
from scripts.supporting.original_main_analysis import run_original_main
from scripts.supporting.presentation import build_tables
from scripts.run import publish_files

main = pd.read_csv(repository / "data/regression_data.csv")
display(main.head())
print(f"Main dataset: {len(main)} observations.")


## 1. Prepare the analysis datasets

We check the main dataset against `data/derived/component_evidence.csv`, which retains the conditions and their coding decisions. We combine conditions using negotiation counts when available and equal weights otherwise. The stricter comparison keeps only the conditions marked for strict inclusion. The combined-payoff comparison averages independent experiment outcomes that used the same payoff record.

These intermediate datasets are built below and saved in `data/derived/`. The [data guide](../data/README.md) explains the coding behind them. To also recalculate the structural measures from the coded payoff and outcome files, change `rebuild_from_coded_sources` below to `True`.


In [ ]:
temporary = TemporaryDirectory(prefix="study1-notebook-")
working = Path(temporary.name)
rebuild_from_coded_sources = False
datasets = (build_data(repository, working) if rebuild_from_coded_sources
            else prepare_analysis_data(repository, working))
display(pd.DataFrame([{"Dataset": name, "Observations": len(frame)}
                      for name, frame in datasets.items()]))


## 2. Run the models and checks

The code fits the five models, repeats them with the stricter data, and runs the negotiation-count weighted and combined-payoff comparisons. Continuous predictors are standardized in each fit. The required resampling checks use 5,000 draws with fixed seeds; the omission checks remove one observation at a time. The original-registration appendix tests are also included.


In [ ]:
run_analysis(datasets, working, draws=5000)
run_original_main(repository, working, draws=5000)
build_tables(repository, working)
publish_files(repository, working)
results = repository / "results"
checks = results / "checks"
print("Saved the intermediate datasets in data/ and the results in results/.")


## 3. Read the tables

Tables 1–3 contain the descriptive statistics, five main regression models, and robustness checks. Tables C1–C4 give the supplementary results. The same tables are saved as separate CSV files in `results/tables/`.


In [ ]:
display(Markdown((results / "tables/README.md").read_text(encoding="utf-8")))


## 4. Check the supporting results

The first table shows whether each hypothesis meets all its criteria. The next tables show the resampling intervals and the range of results after removing individual observations. Every detailed check is saved in `results/checks/`.


In [ ]:
display(pd.read_csv(checks / "hypothesis_tests.csv"))
display(pd.read_csv(checks / "bootstrap_intervals.csv"))
omissions = pd.read_csv(checks / "leave_one_out.csv")
display(omissions.groupby(["dataset", "weighted"]).agg(
    checks=("omitted_observation", "size"),
    smallest_coefficient=("search_complexity_b", "min"),
    largest_coefficient=("search_complexity_b", "max"),
    negative_and_significant=("negative_and_significant", "sum")))
display(pd.read_csv(checks / "original_main_hypotheses.csv"))


## 5. See Figure 2

The figure uses the main dataset loaded above. Figure 1 in the paper is a conceptual illustration.


In [ ]:
display(Image(filename=str(results / "figures/figure_2.png")))
temporary.cleanup()


## Optional: rebuild the dataset from the coded payoff tables

The analyses above start from the released structural measures. To recalculate those measures from the coded payoff tables and outcome sources as well, run this command from the repository's main folder:

```bash
python scripts/run.py --rebuild-data
```

This runs the complete data construction and the same analyses. Sources and coding decisions are described in [data/README.md](../data/README.md). The original papers and licensed teaching materials are not needed to execute the numerical replication.
